In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt


import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole

---

In [ ]:
def check_recursive_feasible_state_pendulum(x_k, omega_max=10.0, T_max=5.0, g=9.81, m=1, l=1):
    theta_k = x_k[0]
    omega_k = x_k[1]

    valid_theta = (jnp.abs(theta_k) <= jnp.pi)

    def true_fun(theta_k):
        return jnp.abs(theta_k)

    def false_fun(theta_k):
        return 2*jnp.pi - jnp.abs(theta_k)

    effective_angle = jax.lax.cond(jnp.sign(theta_k) != jnp.sign(omega_k), true_fun, false_fun, theta_k)

    omega_threshold = jnp.sqrt(omega_max**2 + (4 * g * jnp.cos(theta_k)) / (l) + (2 * T_max * effective_angle) / (m * l**2))
    valid_omega = (jnp.abs(omega_k) <= omega_threshold)

    return jnp.logical_and(valid_theta, valid_omega), omega_threshold

def check_recursive_feasible_action_pendulum(x_k, u_k, omega_threshold, g, m, l, tau):
    theta_k = x_k[0]
    omega_k = x_k[1]
    
    torque_threshold_1 = m*l**2 / tau * (omega_threshold - omega_k) - l*m*g*jnp.sin(theta_k)
    valid_action_1 = (u_k <= torque_threshold_1)
    torque_threshold_2 = m*l**2 / tau * (-omega_threshold - omega_k) - l*m*g*jnp.sin(theta_k)
    valid_action_2 = (u_k >= torque_threshold_2)

    valid_action = jnp.logical_and(valid_action_1, valid_action_2)
    return valid_action, torque_threshold_1, torque_threshold_2

    # omega_testing_value = omega_k + tau * (u_k + l * m * g * jnp.sin(theta_k)) / (m * l**2)
    # valid_action = (jnp.abs(omega_testing_value) <= jnp.abs(omega_threshold))

    # return valid_action, None, None
    

@eqx.filter_jit
def check_recursive_feasible_pendulum(x_k, u_k, tau=2e-2, omega_max=10.0, T_max=5.0, g=9.81, m=1, l=1):
    valid_state, omega_threshold = check_recursive_feasible_state_pendulum(x_k, omega_max=omega_max, T_max=T_max, g=g, m=m, l=l)
    valid_action, torque_threshold_1, torque_threshold_2 = check_recursive_feasible_action_pendulum(x_k, u_k, omega_threshold, g=g, m=m, l=l, tau=tau)
    return jnp.logical_and(valid_state, valid_action), omega_threshold, torque_threshold_1, torque_threshold_2
    # return valid_action, omega_threshold, torque_threshold_1, torque_threshold_2

In [ ]:
thetas = jnp.linspace(-jnp.pi, jnp.pi, 300)
omegas = jnp.linspace(-10, 10, 300)

xx, yy = jnp.meshgrid(thetas, omegas)
x = jnp.stack([xx, yy], axis=-1).reshape(-1, 2)

In [ ]:
out_bool_state, _ = jax.vmap(jax.vmap(check_recursive_feasible_state_pendulum))(jnp.stack([xx, yy], axis=-1))

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(10,10))

ax.imshow(out_bool_state, cmap="plasma", extent=[-np.pi, +np.pi, 10, -10])
ax.set_ylabel(r"$\omega$")
ax.set_xlabel(r"$\theta$")
plt.savefig("recursive_feasible_state_space.png", dpi=400)
plt.show()

In [ ]:
jnp.all(jnp.flipud(jnp.fliplr(out_bool_state)) == out_bool_state)

In [ ]:
thetas = jnp.linspace(-jnp.pi, jnp.pi, 200)
omegas = jnp.linspace(-10, 10, 200)
torques = jnp.linspace(-5, 5, 200)

xx, yy, zz = jnp.meshgrid(thetas, omegas, torques)

In [ ]:
#for tau in jnp.arange(1e-3, 1e-2, 1e-3):

tau = 2e-2
print(tau)
out_bool, omega_threshold, _, _ = jax.vmap(jax.vmap(jax.vmap(check_recursive_feasible_pendulum, in_axes=(0,0,None)), in_axes=(0,0,None)), in_axes=(0,0,None))(jnp.stack([xx, yy], axis=-1), zz, tau)

fig, ax = plt.subplots(1,3, figsize=(10,10), sharey=True)

ax[0].imshow(jnp.any(out_bool, axis=-1), cmap="plasma", extent=[-np.pi, +np.pi, -10, 10], interpolation="nearest")
ax[0].set_ylabel(r"$\omega$")
ax[0].set_xlabel(r"$\theta$")

ax[1].imshow(jnp.all(out_bool, axis=-1), cmap="plasma", extent=[-np.pi, +np.pi, -10, 10], interpolation="nearest")
ax[1].set_xlabel(r"$\theta$")

ax[2].imshow(jnp.logical_xor(jnp.any(out_bool,  axis=-1), jnp.all(out_bool,  axis=-1)), cmap="plasma", extent=[-np.pi, +np.pi, -10, 10],  interpolation="nearest")
ax[2].set_xlabel(r"$\theta$")
# plt.savefig("recursive_feasible_space.png", dpi=400)
fig.tight_layout()
plt.show()

In [ ]:
plt.imshow(jnp.any(out_bool, axis=-1)[:50, :50] == jnp.flipud(jnp.fliplr(jnp.any(out_bool, axis=-1)[-50:, -50:])))

In [ ]:
any_safe = jnp.any(out_bool, axis=-1)
all_safe = jnp.all(out_bool, axis=-1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(40, 10))
plt.imshow(jnp.hstack([any_safe, any_safe, any_safe, any_safe, any_safe, any_safe, any_safe, any_safe]), cmap="plasma", extent=[-np.pi, +7*np.pi, -10, 10], interpolation="nearest")

In [ ]:
any_safe.repeat(2, axis=0).shape

In [ ]:
any_safe_action = jnp.any(out_bool, axis=-1)

In [ ]:
unsafe_positive = (0, 50)
any_safe_action[unsafe_positive]
theta_value_unsafe_positive = jnp.unique(xx[unsafe_positive])
omega_value_unsafe_positive = jnp.unique(yy[unsafe_positive])
print(any_safe_action[unsafe_positive])
print(theta_value_unsafe_positive)
print(omega_value_unsafe_positive)

In [ ]:
unsafe_negative = (-1, 245)

theta_value_unsafe_negative = jnp.unique(xx[unsafe_negative])
omega_value_unsafe_negative = jnp.unique(yy[unsafe_negative])
print(any_safe_action[unsafe_negative])
print(theta_value_unsafe_negative)
print(omega_value_unsafe_negative)

- which actions are unsafe in each case and why?

In [ ]:
print("# Unsafe actions", len(jnp.where(jnp.logical_not(out_bool[unsafe_positive]))[0]))
print("# Safe actions", len(jnp.where(out_bool[unsafe_positive])[0]))

In [ ]:
print("# Unsafe actions", len(jnp.where(jnp.logical_not(out_bool[unsafe_negative]))[0]))
print("# Safe actions", len(jnp.where(out_bool[unsafe_negative])[0]))

In [ ]:
theta_value_unsafe = jnp.unique(xx[0][211])
omega_value_unsafe = jnp.unique(yy[0][211])
unsafe_actions = zz[0][211][jnp.where(jnp.logical_not(out_bool[0][211]))[0]]

allegedly_theta_value_safe = jnp.unique(xx[0][210])
allegedly_omega_value_unsafe = jnp.unique(yy[0][210])
safe_actions = zz[0][211]

In [ ]:
x_unsafe = jnp.array([theta_value_unsafe, omega_value_unsafe])
unsafe_actions
x_unsafe

print("state recursive feasible:", check_recursive_feasible_state_pendulum(x_unsafe))
print("state+action recursive feasible:", jax.vmap(check_recursive_feasible_pendulum, in_axes=(None, 0))(x_unsafe, unsafe_actions)[0])

In [ ]:
x_safe = jnp.array([allegedly_theta_value_safe, allegedly_omega_value_unsafe])
safe_actions
x_safe

print("state recursive feasible:", check_recursive_feasible_state_pendulum(x_safe))
print("state+action recursive feasible:", jax.vmap(check_recursive_feasible_pendulum, in_axes=(None, 0))(x_safe, safe_actions)[0])

In [ ]:
def ode(x, u, m=1, l=1, g=9.81):
    theta = x[0]
    omega = x[1]

    dx1dt = omega
    dx2dt = (u + l * m * g * jnp.sin(theta)) / (m * l**2)
    return jnp.array([dx1dt, dx2dt])

@eqx.filter_jit
def euler_step(x, u, tau):
    return x + tau * ode(x, u)

In [ ]:
eqx.filter_vmap(ode, in_axes=(None, 0))(x_safe, safe_actions)

In [ ]:
eqx.filter_vmap(euler_step, in_axes=(None, 0, None))(x_safe, safe_actions, 2e-2)

- **there does not seem to be a solution for these cases, so why are they part of any?** -> this seems to be an issue with the absolute values or a mistake in the implementation

idx 89 and 90 suprisingly are state+action feasible even though 88 is not -> how can that be? look directly into the computations. Something is going wrong here

In [ ]:
xx[0][88]
yy[0][88]
plt.plot(zz[0][88][jnp.where(jnp.logical_not(out_bool[0][88]))[0]], 'r.')

In [ ]:
xx[0][89]
yy[0][89]

plt.plot(zz[0][89][jnp.where(jnp.logical_not(out_bool[0][89]))[0]], 'r.')

In [ ]:
print("# Unsafe actions", len(jnp.where(jnp.logical_not(out_bool[0][89]))[0]))
print("# Safe actions", len(jnp.where(out_bool[0][89])[0]))

In [ ]:
its in 1. and 2. but not in 3.

1. In which states does a safe action exist under the given conditions
2. In which states does an unsafe action exist
3. In which states does a safe action exist, but also an unsafe action

- checkout all the abs
- where does this slim extra safe space come from? it seems to come from the tau. I am not quite sure yet what the exact reason is

- it might be solvable by using a smaller $\tau$ and taking a higher amount of steps to get back to the original sampling frequency
- Is it possible to solve the equation for arbitrary ODE solvers? It would be preferable to do it for Tsit to properly reflect the simulated dynamics

In [ ]:
jnp.all(out_bool, axis=-1) == out_bool_state
plt.imshow(jnp.all(out_bool, axis=-1) == out_bool_state)

In [ ]:
jnp.all(jnp.any(out_bool, axis=-1) == out_bool_state)
# Makes sense: The R_X is the case where we always apply maximum deceleration. And this is among the options given in R_X,U
# It could only be bigger if there was some choice other than maximum deceleration that was better suited for reducing speed